<a href="https://colab.research.google.com/github/Neha-90/ai-mental-health-chatbot/blob/main/aimentalhealthchatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================
# 1. INSTALL
# =========================
!pip install -q transformers datasets accelerate

# =========================
# 2. IMPORTS
# =========================
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

# =========================
# 3. LOAD DATASET
# =========================
dataset = load_dataset("ShenLab/MentalChat16K")
data = dataset["train"].select(range(300))

# =========================
# 4. LOAD MODEL (FIXED)
# =========================
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

# =========================
# 5. FORMAT DATA
# =========================
def format_data(example):
    text = (
        "You are a kind mental health assistant.\n\n"
        f"User: {example['input']}\n"
        f"Assistant: {example['output']}"
    )
    return {"text": text}

data = data.map(format_data)

# =========================
# 6. TOKENIZE
# =========================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_data = data.map(tokenize, batched=True)

tokenized_data = tokenized_data.remove_columns(
    ["instruction", "input", "output", "text"]
)

# =========================
# 7. TRAINING
# =========================
training_args = TrainingArguments(
    output_dir="./mental_chatbot",
    per_device_train_batch_size=4,
    num_train_epochs=3,
    logging_steps=20,
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data
)

trainer.train()

# =========================
# 8. CHAT FUNCTION
# =========================
def chat(user_input):
    device = model.device

    prompt = (
        "You are a kind mental health assistant.\n\n"
        f"User: {user_input}\nAssistant:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("Assistant:")[-1].strip()

# =========================
# 9. TEST
# =========================
print("Test 1:")
print(chat("I feel anxious"))

print("\nTest 2:")
print(chat("I am feeling very lonely these days"))

print("\nTest 3:")
print(chat("I am stressed about my exams and can't focus"))